In [4]:
import os
import re
import sys
import json
import torch
import subprocess
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from tqdm import tqdm
from torchcrf import CRF

# --- Configuration ---
TRAIN_FILE = "bilstm_data/bilstm_train.jsonl"
VAL_FILE = "bilstm_data/bilstm_validation.jsonl"
TEST_FILE = "bilstm_data/bilstm_test.jsonl"
OUTPUT_DIR = "./bilstm_crf_model"

BATCH_SIZE = 64
EPOCHS = 10           
LEARNING_RATE = 0.001 
WORD_EMBED_DIM = 128
CHAR_EMBED_DIM = 30
CHAR_CNN_FILTERS = 30
HIDDEN_DIM = 256

def get_emptiest_gpu_safely():
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        print("\n🔍 Scanning available GPUs safely via nvidia-smi...")
        # Added utilization.gpu to the query
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,memory.free,utilization.gpu", "--format=csv,nounits,noheader"], 
            encoding="utf-8"
        )
        
        best_id = -1
        max_free_mb = 0
        
        # Fallback variables in case ALL GPUs are >50% utilized
        fallback_id = 0
        fallback_max_mb = 0
        
        for line in result.strip().split("\n"):
            parts = line.split(", ")
            gpu_id = int(parts[0])
            free_memory = int(parts[1])
            gpu_util = int(parts[2])  # Extracted GPU utilization
            
            print(f"  GPU {gpu_id}: {free_memory} MB free | {gpu_util}% util")
            
            # Track the highest memory GPU globally as a fallback
            if free_memory > fallback_max_mb:
                fallback_max_mb = free_memory
                fallback_id = gpu_id
                
            # Primary logic: Only consider GPUs with < 50% utilization
            if gpu_util < 30:
                if free_memory > max_free_mb:
                    max_free_mb = free_memory
                    best_id = gpu_id
                    
        # Final Selection
        if best_id != -1:
            print(f"--> Selected GPU {best_id} with {max_free_mb} MB free VRAM and low utilization.\n")
            return torch.device(f"cuda:{best_id}")
        else:
            print(f"⚠️ All GPUs are highly utilized (>= 30%). Falling back to GPU {fallback_id} with {fallback_max_mb} MB free.\n")
            return torch.device(f"cuda:{fallback_id}")
            
    except Exception as e:
        print(f"⚠️ Failed to query nvidia-smi: {e}. Falling back to cuda:0.")
        return torch.device("cuda:0")

# --- 1. Data Alignment & Tokenization ---
def reconstruct_char_labels(input_text, output_dict):
    char_labels = ["O"] * len(input_text)
    
    field_to_tag = {
        "flat": "UNIT",
        "floor": "FLOOR",
        "block": "BLOCK",             # <-- ADD THIS
        "phase": "PHASE",             # <-- ADD THIS
        "building_name": "BUILDING_NAME",
        "estate_name": "ESTATE_NAME",
        "street_name": "STREET_NAME",
        "sub_district": "SUB_DISTRICT",
        "district": "LOCATION",
        "region": "REGION",
        "village_name": "VILLAGE_NAME", 
        "building_number": "BUILDING_NUMBER"
    }
    
    # Flatten line1 and line2 into a single dictionary
    flat_targets = {}
    if "line1" in output_dict:
        flat_targets.update(output_dict.get("line1", {}))
        flat_targets.update(output_dict.get("line2", {}))
    else:
        flat_targets = output_dict

    # 1. Gather all terms to process
    items_to_process = []
    for field, tag in field_to_tag.items():
        val = flat_targets.get(field, "")
        if not val:
            continue
            
        search_terms = val.split(" / ") if " / " in val else [val]
        for term in search_terms:
            if term:
                items_to_process.append((field, tag, term))
                
    # 2. CRITICAL FIX: Sort by length of term descending. 
    # This ensures "MUI WO KAU TSUEN" is processed before "MUI WO"
    items_to_process.sort(key=lambda x: len(x[2]), reverse=True)
    
    # 3. Apply tags while avoiding overwriting
    for field, tag, term in items_to_process:
        start_idx = 0
        while True:
            idx = input_text.find(term, start_idx)
            if idx == -1:
                break # Term not found or no more occurrences
            
            # Check if this specific occurrence is already tagged by a longer entity
            is_already_tagged = any(char_labels[i] != "O" for i in range(idx, idx + len(term)))
            
            if not is_already_tagged:
                # Mark B- and I- tags on this specific untagged span
                char_labels[idx] = f"B-{tag}"
                for i in range(idx + 1, idx + len(term)):
                    if i < len(char_labels):
                        char_labels[i] = f"I-{tag}"
                break # Successfully tagged, move to the next term in items_to_process
            else:
                # This occurrence was already tagged, keep searching forward!
                start_idx = idx + 1
                
    return char_labels

def tokenize_and_align(input_text, output_dict):
    """Splits text into English words and Chinese chars, aligning the tags."""
    char_labels = reconstruct_char_labels(input_text, output_dict)
    tokens, token_tags = [], []
    
    # Regex: Matches English/Numbers natively, Chinese chars individually, and symbols
    for match in re.finditer(r'[a-zA-Z0-9]+|[\u4e00-\u9fff]|[^\s]', input_text):
        token_str = match.group()
        start_idx = match.start()
        
        tokens.append(token_str)
        # The tag for the entire word/token is based on its first character
        token_tags.append(char_labels[start_idx])
        
    return tokens, token_tags

# --- 2. Vocab Builder ---
class Vocab:
    def __init__(self):
        self.w2i = {"<PAD>": 0, "<UNK>": 1}
        self.i2w = {0: "<PAD>", 1: "<UNK>"}
        
    def add(self, word):
        if word not in self.w2i:
            idx = len(self.w2i)
            self.w2i[word] = idx
            self.i2w[idx] = word
            
    def __len__(self):
        return len(self.w2i)

# --- 3. PyTorch Dataset & Dataloader ---
class AddressDataset(Dataset):
    def __init__(self, data_list, word_vocab, char_vocab, tag2idx):
        self.data = data_list
        self.w2i = word_vocab
        self.c2i = char_vocab
        self.t2i = tag2idx
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        tokens, tags = self.data[idx]
        
        word_ids = [self.w2i.w2i.get(t, self.w2i.w2i["<UNK>"]) for t in tokens]
        char_ids_list = [[self.c2i.w2i.get(c, self.c2i.w2i["<UNK>"]) for c in token] for token in tokens]
        tag_ids = [self.t2i[tag] for tag in tags]
        
        return word_ids, char_ids_list, tag_ids

def collate_fn(batch):
    max_seq_len = max(len(item[0]) for item in batch)
    max_word_len = max([1] + [len(c) for item in batch for c in item[1]])
    
    b_words, b_chars, b_labels, b_masks = [], [], [], []
    
    for word_ids, char_ids_list, label_ids in batch:
        seq_len = len(word_ids)
        b_words.append(word_ids + [0] * (max_seq_len - seq_len))
        b_labels.append(label_ids + [0] * (max_seq_len - seq_len))
        b_masks.append([True] * seq_len + [False] * (max_seq_len - seq_len)) 
        
        padded_chars = [chars + [0] * (max_word_len - len(chars)) for chars in char_ids_list]
        padded_chars.extend([[0] * max_word_len for _ in range(max_seq_len - seq_len)])
        b_chars.append(padded_chars)
        
    return {
        "word_ids": torch.tensor(b_words, dtype=torch.long),
        "char_ids": torch.tensor(b_chars, dtype=torch.long),
        "labels": torch.tensor(b_labels, dtype=torch.long),
        "mask": torch.tensor(b_masks, dtype=torch.bool)
    }

# --- 4. The Neural Architecture ---
class BiLSTM_CNN_CRF(nn.Module):
    def __init__(self, vocab_size, char_vocab_size, num_tags, word_dim, char_dim, cnn_filters, hidden_dim, dropout=0.5):
        super().__init__()
        
        # Word Embeddings
        self.word_embed = nn.Embedding(vocab_size, word_dim, padding_idx=0)
        
        # Character Embeddings & CNN
        self.char_embed = nn.Embedding(char_vocab_size, char_dim, padding_idx=0)
        # Padding=1 prevents convolution crashes on 1-character tokens (like Chinese chars)
        self.char_cnn = nn.Conv1d(in_channels=char_dim, out_channels=cnn_filters, kernel_size=3, padding=1)
        
        # BiLSTM
        lstm_input_dim = word_dim + cnn_filters
        self.lstm = nn.LSTM(lstm_input_dim, hidden_dim // 2, num_layers=1, bidirectional=True, batch_first=True)
        
        # Projection & CRF (lexicon features removed from input dim)
        self.dropout = nn.Dropout(dropout)
        self.hidden2tag = nn.Linear(hidden_dim, num_tags)
        self.crf = CRF(num_tags, batch_first=True)
        
    def forward(self, word_ids, char_ids, mask, labels=None):
        batch_size, seq_len = word_ids.shape
        max_word_len = char_ids.shape[2]
        
        # 1. Word Vectors
        w_emb = self.word_embed(word_ids) # shape: (batch, seq, word_dim)
        
        # 2. Character Features (CNN)
        char_ids_flat = char_ids.view(-1, max_word_len) 
        c_emb = self.char_embed(char_ids_flat).permute(0, 2, 1) # reshape for PyTorch Conv1d
        
        c_cnn_out, _ = torch.max(self.char_cnn(c_emb), dim=2)   # Max pooling over the word
        c_features = c_cnn_out.view(batch_size, seq_len, -1)    # shape: (batch, seq, cnn_filters)
        
        # 3. Concatenate and run BiLSTM
        lstm_in = self.dropout(torch.cat([w_emb, c_features], dim=2))
        lstm_out, _ = self.lstm(lstm_in)
        
        # 4. Generate Logits directly from LSTM output
        emissions = self.hidden2tag(lstm_out)
        
        # 5. CRF Decoding / Loss
        if labels is not None:
            return -self.crf(emissions, tags=labels, mask=mask, reduction='mean')
        return self.crf.decode(emissions, mask=mask)

# --- 5. Main Training Pipeline ---
def main():
    device = get_emptiest_gpu_safely()
    
    tags = ["O"]
    # --- UPDATE THIS LIST ---
    tag_list = ["UNIT", "FLOOR", "BUILDING_NAME", "ESTATE_NAME", "STREET_NAME", 
                "SUB_DISTRICT", "LOCATION", "REGION", "VILLAGE_NAME", "BUILDING_NUMBER",
                "BLOCK", "PHASE"]
                
    for t in tag_list:
        tags.extend([f"B-{t}", f"I-{t}"])
    tag2idx = {t: i for i, t in enumerate(tags)}
    
    word_vocab = Vocab()
    char_vocab = Vocab()
    
    def parse_file(file_path, update_vocab=False):
        data_list = []
        if not os.path.exists(file_path):
            return data_list
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                data = json.loads(line)
                tokens, token_tags = tokenize_and_align(data["input"], data["output"])
                if update_vocab:
                    for token in tokens:
                        word_vocab.add(token)
                        for char in token:
                            char_vocab.add(char)
                data_list.append((tokens, token_tags))
        return data_list

    print("📂 Parsing datasets and building Vocabularies...")
    train_data = parse_file(TRAIN_FILE, update_vocab=True)
    val_data = parse_file(VAL_FILE, update_vocab=False)

    # --- ADD THIS CHECK ---
    if len(train_data) == 0:
        raise FileNotFoundError(f"🚨 No training data found! Please ensure '{TRAIN_FILE}' exists and is not empty.")
    # ----------------------

    print(f"✅ Vocab sizes -> Words: {len(word_vocab)}, Chars: {len(char_vocab)}")

    train_loader = DataLoader(AddressDataset(train_data, word_vocab, char_vocab, tag2idx), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(AddressDataset(val_data, word_vocab, char_vocab, tag2idx), batch_size=BATCH_SIZE*2, shuffle=False, collate_fn=collate_fn) if val_data else None

    print("⏳ Initializing NER Architecture...")
    model = BiLSTM_CNN_CRF(
        vocab_size=len(word_vocab), 
        char_vocab_size=len(char_vocab), 
        num_tags=len(tag2idx),
        word_dim=WORD_EMBED_DIM, 
        char_dim=CHAR_EMBED_DIM, 
        cnn_filters=CHAR_CNN_FILTERS, 
        hidden_dim=HIDDEN_DIM
    ).to(device)
    
    optimizer = Adam(model.parameters(), lr=LEARNING_RATE)
    best_val_loss = float('inf')

    print("\n🚀 Training Initiated...")
    
    try:
        for epoch in range(EPOCHS):
            model.train()
            total_train_loss = 0
            
            # Print epoch start
            print(f"--- Epoch {epoch+1}/{EPOCHS} ---")
            
            for step, batch in enumerate(train_loader):
                word_ids, char_ids = batch["word_ids"].to(device), batch["char_ids"].to(device)
                labels, mask = batch["labels"].to(device), batch["mask"].to(device)
                
                optimizer.zero_grad()
                loss = model(word_ids, char_ids, mask, labels=labels)
                loss.backward()
                optimizer.step()
                
                total_train_loss += loss.item()
                
            avg_train_loss = total_train_loss / len(train_loader)

            # Validation Step
            if val_loader:
                model.eval()
                total_val_loss = 0
                with torch.no_grad():
                    for batch in val_loader:
                        word_ids, char_ids = batch["word_ids"].to(device), batch["char_ids"].to(device)
                        labels, mask = batch["labels"].to(device), batch["mask"].to(device)
                        val_loss = model(word_ids, char_ids, mask, labels=labels)
                        total_val_loss += val_loss.item()
                
                avg_val_loss = total_val_loss / len(val_loader)
                
                # Print single line summary for the epoch
                print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

                if avg_val_loss < best_val_loss:
                    best_val_loss = avg_val_loss
                    print(f"🌟 New best validation loss! Exporting to {OUTPUT_DIR}...")
                    os.makedirs(OUTPUT_DIR, exist_ok=True)
                    torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "pytorch_model.bin"))
                    with open(os.path.join(OUTPUT_DIR, "vocabs.json"), "w", encoding="utf-8") as f:
                        json.dump({"w2i": word_vocab.w2i, "c2i": char_vocab.w2i, "t2i": tag2idx}, f)
            else:
                print(f"Train Loss: {avg_train_loss:.4f}")

        print(f"\n✅ Training complete. Best model saved to {OUTPUT_DIR}")

    except KeyboardInterrupt:
        print("\n\n⚠️ Training Interrupted by User (Ctrl+C)!")
        print("👋 Safe exit complete.")
        sys.exit(0)

if __name__ == "__main__":
    main()


🔍 Scanning available GPUs safely via nvidia-smi...
  GPU 0: 10325 MB free | 100% util
  GPU 1: 21520 MB free | 0% util
  GPU 2: 14834 MB free | 99% util
  GPU 3: 6400 MB free | 0% util
--> Selected GPU 1 with 21520 MB free VRAM and low utilization.

📂 Parsing datasets and building Vocabularies...
✅ Vocab sizes -> Words: 31839, Chars: 2856
⏳ Initializing NER Architecture...

🚀 Training Initiated...
--- Epoch 1/10 ---
Train Loss: 4.8444 | Val Loss: 0.9915
🌟 New best validation loss! Exporting to ./bilstm_crf_model...
--- Epoch 2/10 ---
Train Loss: 0.9847 | Val Loss: 0.5661
🌟 New best validation loss! Exporting to ./bilstm_crf_model...
--- Epoch 3/10 ---
Train Loss: 0.6441 | Val Loss: 0.4301
🌟 New best validation loss! Exporting to ./bilstm_crf_model...
--- Epoch 4/10 ---
Train Loss: 0.4997 | Val Loss: 0.3640
🌟 New best validation loss! Exporting to ./bilstm_crf_model...
--- Epoch 5/10 ---
Train Loss: 0.4281 | Val Loss: 0.3435
🌟 New best validation loss! Exporting to ./bilstm_crf_model..

In [6]:
import os
import re
import json
import time
import torch
import subprocess
import torch.nn as nn
from collections import defaultdict
from tqdm import tqdm
from torchcrf import CRF

# ==========================================
# CONFIGURATION
# ==========================================
MODEL_DIR = "./bilstm_crf_model" 
LOG_FILE = "parsing_results_bilstm.log"
TEST_FILE = "bilstm_data/bilstm_test.jsonl" # Pointing to the new test JSONL
# TEST_FILE = "bilstm_data/out_test.jsonl" # Pointing to the new test JSONL

# Must match training script exactly
WORD_EMBED_DIM = 128
CHAR_EMBED_DIM = 30
CHAR_CNN_FILTERS = 30
HIDDEN_DIM = 256
# ==========================================

# ==========================================
# CUSTOM ARCHITECTURE (Lexicon Removed to Match Training)
# ==========================================
class BiLSTM_CNN_CRF(nn.Module):
    def __init__(self, vocab_size, char_vocab_size, num_tags, word_dim, char_dim, cnn_filters, hidden_dim, dropout=0.5):
        super().__init__()
        self.word_embed = nn.Embedding(vocab_size, word_dim, padding_idx=0)
        self.char_embed = nn.Embedding(char_vocab_size, char_dim, padding_idx=0)
        self.char_cnn = nn.Conv1d(in_channels=char_dim, out_channels=cnn_filters, kernel_size=3, padding=1)
        
        lstm_input_dim = word_dim + cnn_filters
        self.lstm = nn.LSTM(lstm_input_dim, hidden_dim // 2, num_layers=1, bidirectional=True, batch_first=True)
        
        self.dropout = nn.Dropout(dropout)
        self.hidden2tag = nn.Linear(hidden_dim, num_tags) # Lexicon feature dimension removed
        self.crf = CRF(num_tags, batch_first=True)
        
    def forward(self, word_ids, char_ids, mask, labels=None):
        batch_size, seq_len = word_ids.shape
        max_word_len = char_ids.shape[2]
        
        w_emb = self.word_embed(word_ids) 
        
        char_ids_flat = char_ids.view(-1, max_word_len) 
        c_emb = self.char_embed(char_ids_flat).permute(0, 2, 1) 
        
        c_cnn_out, _ = torch.max(self.char_cnn(c_emb), dim=2)   
        c_features = c_cnn_out.view(batch_size, seq_len, -1)    
        
        lstm_in = self.dropout(torch.cat([w_emb, c_features], dim=2))
        lstm_out, _ = self.lstm(lstm_in)
        
        emissions = self.hidden2tag(lstm_out)
        
        if labels is not None:
            return -self.crf(emissions, tags=labels, mask=mask, reduction='mean')
        return self.crf.decode(emissions, mask=mask)

# ==========================================
# UTILITY FUNCTIONS
# ==========================================
def get_emptiest_gpu_safely():
    if not torch.cuda.is_available():
        print("CUDA not available. Falling back to CPU (-1).")
        return -1
        
    try:
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,memory.free", "--format=csv,nounits,noheader"], 
            encoding="utf-8"
        )
        best_id, max_free_mb = 0, -1
        for line in result.strip().split("\n"):
            if not line.strip(): continue
            gpu_id, free_memory = line.split(", ")
            gpu_id, free_memory = int(gpu_id), int(free_memory)
            
            if free_memory > max_free_mb:
                max_free_mb = free_memory
                best_id = gpu_id
                
        return best_id
    except Exception:
        return 0

def extract_3d_components(parsed_entities):
    components = defaultdict(list)
    for entity in parsed_entities:
        tag = entity['entity_group']
        if tag == 'O': 
            continue
            
        word = entity['word'].strip()
        if word:
            components[tag].append(word)

    formatted_output = {}
    for tag, words in components.items():
        joined_string = "".join(words)
        if any('\u4e00' <= char <= '\u9fff' for char in joined_string):
            # Chinese logic remains unchanged
            formatted_output[tag] = "".join(words) 
        else:
            # English logic: Join with spaces, then fix spacing around punctuation
            joined_en = " ".join(words).strip()
            # This removes spaces immediately before and after a slash, dot or hyphen
            joined_en = re.sub(r'\s*([/\.-])\s*', r'\1', joined_en)
            formatted_output[tag] = joined_en
            
    return formatted_output

def assemble_compact_json(extracted_data):
    return {
        "line1": {
            "flat": extracted_data.get("UNIT", ""),
            "floor": extracted_data.get("FLOOR", ""),
            "block": extracted_data.get("BLOCK", ""),       # <-- ADD THIS
            "phase": extracted_data.get("PHASE", ""),       # <-- ADD THIS
            "building_name": extracted_data.get("BUILDING_NAME", "")
        },
        "line2": {
            "estate_name": extracted_data.get("ESTATE_NAME", ""),
            "village_name": extracted_data.get("VILLAGE_NAME", ""),       
            "building_number": extracted_data.get("BUILDING_NUMBER", ""), 
            "street_name": extracted_data.get("STREET_NAME", ""),
            "sub_district": extracted_data.get("SUB_DISTRICT", ""), 
            "district": extracted_data.get("LOCATION", ""),
            "region": extracted_data.get("REGION", "")
        }
    }

def flatten_json(output_dict):
    """Flattens ground truth or prediction nested dicts into a single level dictionary for easy comparison."""
    flat = {}
    if "line1" in output_dict:
        flat.update(output_dict.get("line1", {}))
        flat.update(output_dict.get("line2", {}))
    else:
        flat = output_dict
    return {k: v for k, v in flat.items() if v}

# ==========================================
# MAIN EXECUTION
# ==========================================
def main():
    total_time = 0
    device_id = get_emptiest_gpu_safely()
    device = torch.device(f"cuda:{device_id}" if device_id != -1 else "cpu")
    
    print(f"DEBUG: Loading Vocabs and BiLSTM+CNN+CRF Model from {MODEL_DIR}...")
    
    vocab_path = os.path.join(MODEL_DIR, "vocabs.json")
    if not os.path.exists(vocab_path):
        print(f"❌ Error: {vocab_path} not found.")
        return
        
    with open(vocab_path, "r", encoding="utf-8") as f:
        vocabs = json.load(f)
        
    w2i = vocabs.get("w2i", {})
    c2i = vocabs.get("c2i", {})
    t2i = vocabs.get("t2i", {})
    idx2tag = {int(v): k for k, v in t2i.items()} 
    
    UNK_W_ID = w2i.get("<UNK>", 1)
    UNK_C_ID = c2i.get("<UNK>", 1)
    
    model = BiLSTM_CNN_CRF(
        vocab_size=len(w2i), 
        char_vocab_size=len(c2i),
        num_tags=len(t2i),
        word_dim=WORD_EMBED_DIM,
        char_dim=CHAR_EMBED_DIM,
        cnn_filters=CHAR_CNN_FILTERS,
        hidden_dim=HIDDEN_DIM,
        dropout=0.0 # No dropout during inference
    )
    
    weights_path = os.path.join(MODEL_DIR, "pytorch_model.bin")
    if os.path.exists(weights_path):
        model.load_state_dict(torch.load(weights_path, map_location=device))
    else:
        print(f"⚠️ Warning: Weights not found at {weights_path}.")
    
    model.to(device)
    model.eval()

    if not os.path.exists(TEST_FILE):
        print(f"❌ Error: Dataset file '{TEST_FILE}' not found.")
        return

    # Tracking Metrics
    all_fields = [
        "flat", "floor", "building_name", 
        "block", "phase",                             # <-- ADD THEM HERE
        "estate_name", "village_name", "building_number",
        "street_name", "sub_district", "district", "region"
    ]
    total_samples = 0
    exact_matches = 0
    field_correct = {field: 0 for field in all_fields}
    field_totals = {field: 0 for field in all_fields} # Tracks how many times a field actually existed in GT or Pred

    print("🚀 Running evaluation and accuracy check...")
    with open(TEST_FILE, "r", encoding="utf-8") as file, \
         open(LOG_FILE, "w", encoding="utf-8") as log:
        
        lines = [line for line in file if line.strip() and not line.startswith('#')]
        
        for line in tqdm(lines, desc="Evaluating"):
            data = json.loads(line)
            address = data["input"].strip()
            ground_truth_raw = data.get("output", {})
            ground_truth_flat = flatten_json(ground_truth_raw)
            
            start_time = time.perf_counter()
                
            # --- TOKENIZATION ---
            tokens = []
            for match in re.finditer(r'[a-zA-Z0-9]+|[\u4e00-\u9fff]|[^\s]', address):
                tokens.append(match.group())
                
            if not tokens:
                continue
                
            # --- CONVERT TO TENSORS ---
            word_ids = [w2i.get(t, UNK_W_ID) for t in tokens]
            max_word_len = max([1] + [len(t) for t in tokens])
            
            char_ids = []
            for t in tokens:
                c_ids = [c2i.get(c, UNK_C_ID) for c in t]
                c_ids += [0] * (max_word_len - len(c_ids)) # Pad characters
                char_ids.append(c_ids)
                
            word_tensor = torch.tensor([word_ids], dtype=torch.long).to(device)
            char_tensor = torch.tensor([char_ids], dtype=torch.long).to(device)
            mask_tensor = torch.tensor([[True] * len(tokens)], dtype=torch.bool).to(device)

            # --- INFERENCE ---
            with torch.no_grad():
                prediction_ids = model(word_tensor, char_tensor, mask_tensor)[0]
            
            # --- TAG ALIGNMENT ---
            parsed_entities = []
            for token_str, tag_id in zip(tokens, prediction_ids):
                tag = idx2tag[tag_id]
                entity_group = tag.replace("B-", "").replace("I-", "")
                if tag == "O": 
                    entity_group = "O"
                    
                parsed_entities.append({"entity_group": entity_group, "word": token_str})
            
            extracted_data = extract_3d_components(parsed_entities)
            final_json = assemble_compact_json(extracted_data)
            predicted_flat = flatten_json(final_json)
            
            total_time += (time.perf_counter() - start_time)

            # --- ACCURACY COMPARISON ---
            total_samples += 1
            is_perfect_match = True
            
            for field in all_fields:
                pred_val = predicted_flat.get(field, "")
                gt_val = ground_truth_flat.get(field, "")
                
                # Track correct predictions
                if pred_val == gt_val:
                    field_correct[field] += 1
                else:
                    is_perfect_match = False
                    
            if is_perfect_match:
                exact_matches += 1

            # --- LOGGING ---
            log.write(f"Original: {address}\n")
            if is_perfect_match:
                log.write("✅ EXACT MATCH\n")
            else:
                log.write("❌ MISMATCH FOUND\n")
                
            for field in all_fields:
                pred_val = predicted_flat.get(field, "")
                gt_val = ground_truth_flat.get(field, "")
                if pred_val or gt_val:
                    status = "✅" if pred_val == gt_val else "❌"
                    log.write(f"  {status} {field.upper()}:\n")
                    log.write(f"      PRED: {pred_val if pred_val else '[None]'}\n")
                    log.write(f"      TRUE: {gt_val if gt_val else '[None]'}\n")
            log.write("-" * 50 + "\n")

    # --- PRINT ACCURACY REPORT ---
    print("\n" + "="*40)
    print("📊 EVALUATION RESULTS")
    print("="*40)
    print(f"Total Samples Tested : {total_samples}")
    if total_samples > 0:
        exact_match_acc = (exact_matches / total_samples) * 100
        print(f"Exact Match Accuracy : {exact_match_acc:.2f}% ({exact_matches}/{total_samples} perfect addresses)")
        print("\n--- Field-Level Accuracy ---")
        for field in all_fields:
            acc = (field_correct[field] / total_samples) * 100
            print(f"{field.rjust(15)} : {acc:.2f}% ({field_correct[field]}/{total_samples})")
    
    print("="*40)
    print(f"✅ Processing complete. Detailed line-by-line results saved to {LOG_FILE}")
    print(f"⏱️ Total Inference runtime: {total_time:.4f} seconds")

if __name__ == "__main__":
    main()

DEBUG: Loading Vocabs and BiLSTM+CNN+CRF Model from ./bilstm_crf_model...
🚀 Running evaluation and accuracy check...


Evaluating: 100%|██████████| 20814/20814 [01:52<00:00, 185.26it/s]



📊 EVALUATION RESULTS
Total Samples Tested : 20814
Exact Match Accuracy : 69.85% (14539/20814 perfect addresses)

--- Field-Level Accuracy ---
           flat : 93.16% (19391/20814)
          floor : 89.06% (18537/20814)
  building_name : 95.95% (19970/20814)
          block : 99.36% (20681/20814)
          phase : 99.29% (20666/20814)
    estate_name : 96.48% (20082/20814)
   village_name : 98.31% (20462/20814)
building_number : 97.42% (20276/20814)
    street_name : 98.51% (20504/20814)
   sub_district : 96.08% (19999/20814)
       district : 97.11% (20213/20814)
         region : 96.03% (19987/20814)
✅ Processing complete. Detailed line-by-line results saved to parsing_results_bilstm.log
⏱️ Total Inference runtime: 108.1753 seconds
